In [2]:
# 1. Install dependencies + accelerate
!pip install -q streamlit sentence-transformers transformers tokenizers pandas numpy accelerate
# 2. Install localtunnel
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 122.0 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 4s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙

In [3]:
# 1. Clean up any existing directory if running again
!rm -rf economic-news-sentiment

# 2. Clone your clean GitHub repo (replace with your personal GitHub URL)
!git clone https://github.com/Nas-Mohd/economic-news-sentiment.git

# 3. CRITICAL: Move your notebook's working directory into the directory containing app.py
# Use '%cd' (IPython magic) instead of '!cd' (subshell) to keep the working directory active!
%cd economic-news-sentiment/demo

Cloning into 'economic-news-sentiment'...
remote: Enumerating objects: 652, done.
remote: Counting objects: 100% (269/269), done.
remote: Compressing objects: 100% (201/201), done.
remote: Total 652 (delta 155), reused 156 (delta 68), pack-reused 383 (from 1)
Receiving objects: 100% (652/652), 8.82 MiB | 5.59 MiB/s, done.
Resolving deltas: 100% (398/398), done.
/content/economic-news-sentiment/demo


In [5]:
!pip install gradio transformers torch pandas numpy beautifulsoup4 -q
# 3. Secure drive connection, open port, and boot UI
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
# =====================================================================
# 📦 STEP 1: INITIALIZE DEPENDENCIES & EXPERT CALIBRATED THRESHOLDS
# =====================================================================
import os
import re
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
import gradio as gr
from bs4 import BeautifulSoup
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Core Configuration Profiles
ASPECTS = ["Monetary Financial", "Inflation Prices", "Real Economic Activity", "Labor Consumption", "Fiscal Government", "External Sector"]
LABEL_MAPPING = {0: "🟢 Positive", 1: "🔴 Negative", 2: "🟡 Neutral"}

# YOUR EXACT CALIBRATED DEBERTA THRESHOLDS MATRIX
DEBERTA_THRESHOLDS = {
    "Monetary Financial": 0.45,
    "Inflation Prices": 0.40,
    "Real Economic Activity": 0.30,
    "Labor Consumption": 0.25,
    "Fiscal Government": 0.35,
    "External Sector": 0.25
}

# 🌐 CLEAN REPOSITORY ROUTING (REPLACED DRIVE STORAGE WITH HF HUB LINKS)
DEBERTA_REPO = "dummfak/deberta-v3-base-macroeconomic-aspect-classifier"
FINBERT_TUNED_REPO = "dummfak/finbert-macroeconomic-absa"
BASE_MODEL_NAME = "ProsusAI/finbert"

print("📥 Initializing deep learning architectures straight from Hugging Face Hub...")

# Determine execution device dynamically (GPU if available, else CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Routing computing matrix to: {device.upper()}")

def load_classification_assets(path_or_name, is_deberta=False):
    try:
        if is_deberta:
            tok = AutoTokenizer.from_pretrained(path_or_name, use_fast=False)
        else:
            tok = AutoTokenizer.from_pretrained(path_or_name)

        # Explicitly configure output_hidden_states=True on base model for vector embeddings
        if path_or_name == BASE_MODEL_NAME:
            mod = AutoModelForSequenceClassification.from_pretrained(path_or_name, output_hidden_states=True)
        else:
            mod = AutoModelForSequenceClassification.from_pretrained(path_or_name)

        mod = mod.to(device)
        mod.eval()
        return tok, mod
    except Exception as e:
        print(f"⚠️ Repo asset unavailable ({path_or_name}). Falling back to baseline architecture. Error: {e}")
        tok = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
        mod = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_NAME, output_hidden_states=True)
        mod = mod.to(device)
        mod.eval()
        return tok, mod

# Initialize Model Instances from Hugging Face Hub
base_tok, base_mod = load_classification_assets(BASE_MODEL_NAME, is_deberta=False)
deb_tok, deb_mod = load_classification_assets(DEBERTA_REPO, is_deberta=True)
abs_tok, abs_mod = load_classification_assets(FINBERT_TUNED_REPO, is_deberta=False)

print("🚀 Core neural engines loaded successfully!")

# =====================================================================
# 🧠 STEP 2: ENGINE COMPUTATION LOGIC SHARDS
# =====================================================================

def run_inference(tokenizer, model, text, text_pair=None):
    """Passes strings cleanly and returns pure raw unscaled NumPy logits"""
    with torch.no_grad():
        inputs = tokenizer(
            text=str(text).strip(),
            text_pair=text_pair,
            max_length=128,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).to(device)

        outputs = model(**inputs)
        raw_logits = outputs.logits.squeeze(0).cpu().numpy()

    return raw_logits

def get_semantic_embedding(text):
    """Generates a mean-pooled sentence embedding utilizing FinBERT's hidden states"""
    with torch.no_grad():
        inputs = base_tok(
            str(text).strip(),
            max_length=128,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).to(device)

        outputs = base_mod(**inputs)
        # Extract last hidden state layer: shape [1, seq_len, hidden_dim]
        last_hidden_state = outputs.hidden_states[-1]

        # Mean Pooling across the sequence length, ignoring padding tokens via attention mask
        attention_mask = inputs['attention_mask'].unsqueeze(-1) # [1, seq_len, 1]
        mask_embeddings = last_hidden_state * attention_mask
        sum_embeddings = torch.sum(mask_embeddings, dim=1)
        sum_mask = torch.clamp(attention_mask.sum(dim=1), min=1e-9)

        mean_pooled = (sum_embeddings / sum_mask).squeeze(0) # [hidden_dim]
    return mean_pooled

def process_routing(sentence):
    """Page 2: Replicates your training notebook's Sigmoid conversion perfectly"""
    if sentence is None or not str(sentence).strip():
        return pd.DataFrame(columns=["Macro Aspect Group", "Pipeline Classification Status"])

    clean_sentence = str(sentence).strip()

    # 1. Hybrid Baseline: Calculate Keyword Matches AND Semantic Cosine Similarity
    sentence_vector = get_semantic_embedding(clean_sentence)

    baseline_reports = {}
    for aspect in ASPECTS:
        # A. Keyword Matching Logic
        keywords = set(re.sub(r'[^a-zA-Z ]', '', aspect.lower()).split())
        match_count = sum(1 for word in keywords if word in clean_sentence.lower())

        # B. Semantic Similarity Logic
        aspect_vector = get_semantic_embedding(aspect)
        cosine_sim = F.cosine_similarity(sentence_vector.unsqueeze(0), aspect_vector.unsqueeze(0)).item()

        # Combined baseline text feedback
        baseline_reports[aspect] = f"Matches: {match_count} | Sim: {cosine_sim:.3f}"

    # 2. Fine-Tuned Route: Pull raw logits and run manual Sigmoid scaling
    raw_logits = run_inference(deb_tok, deb_mod, clean_sentence)
    deb_probs = 1 / (1 + np.exp(-raw_logits))

    output_df = []
    for i, aspect in enumerate(ASPECTS):
        prob_val = float(deb_probs[i]) if i < len(deb_probs) else 0.0

        threshold = DEBERTA_THRESHOLDS[aspect]
        status = "🟢 [ACTIVE]" if prob_val >= threshold else "⚪ [inactive]"

        output_df.append({
            "Macro Aspect Group": aspect,
            "Baseline (Keywords & Semantics)": baseline_reports[aspect],
            "DeBERTa Sigmoid Score": f"{prob_val * 100:.1f}%",
            "Assigned Optimal Threshold": f"{threshold * 100:.1f}%",
            "Pipeline Classification Status": status
        })

    return pd.DataFrame(output_df)

def process_sentiment(sentence, aspect):
    """Page 3: Evaluates Context Isolation using side-by-side validation"""
    if sentence is None or aspect is None or not str(sentence).strip():
        return {"⚠️ Awaiting Input": 1.0}, {"⚠️ Awaiting Input": 1.0}

    clean_sentence = str(sentence).strip()
    clean_aspect = str(aspect).strip()

    # Left Column: Standard Base FinBERT
    base_logits = run_inference(base_tok, base_mod, clean_sentence)
    base_probs = F.softmax(torch.tensor(base_logits), dim=-1).numpy()
    base_res = {LABEL_MAPPING[idx]: float(base_probs[idx]) for idx in range(min(3, len(base_probs)))}

    # Right Column: Your Custom Fine-Tuned ABSA Model
    tuned_logits = run_inference(abs_tok, abs_mod, clean_sentence, text_pair=clean_aspect)
    tuned_probs = F.softmax(torch.tensor(tuned_logits), dim=-1).numpy()
    tuned_res = {LABEL_MAPPING[idx]: float(tuned_probs[idx]) for idx in range(min(3, len(tuned_probs)))}

    return base_res, tuned_res

def process_document(raw_doc):
    """Page 4: Sentence-centric Matrix Compilation Log"""
    if raw_doc is None or not str(raw_doc).strip():
        return pd.DataFrame(columns=["Parsed Sentence Context"] + ASPECTS)

    str_doc = str(raw_doc).strip()

    # Strip HTML formatting cleanly if present
    if "<html" in str_doc.lower() or "<div>" in str_doc.lower() or "<p>" in str_doc.lower():
        soup = BeautifulSoup(str_doc, "html.parser")
        clean_text = soup.get_text(separator=" ")
    else:
        clean_text = str_doc

    # Split text blocks into independent clean evaluation sentences
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', clean_text) if len(s.strip()) > 10]

    records = []
    for sent in sentences:
        raw_deb_logits = run_inference(deb_tok, deb_mod, sent)
        deb_probs = 1 / (1 + np.exp(-raw_deb_logits))

        # Row generation anchor centered completely around the unique sentence context
        row_record = {"Parsed Sentence Context": sent}

        for idx, aspect in enumerate(ASPECTS):
            prob = float(deb_probs[idx]) if idx < len(deb_probs) else 0.0
            threshold = DEBERTA_THRESHOLDS[aspect]

            if prob >= threshold:
                raw_sent_logits = run_inference(abs_tok, abs_mod, sent, text_pair=aspect)
                sent_probs = F.softmax(torch.tensor(raw_sent_logits), dim=-1).numpy()
                winning_idx = np.argmax(sent_probs)

                sentiment_label = LABEL_MAPPING[winning_idx]
                confidence_score = sent_probs[winning_idx] * 100

                row_record[aspect] = f"{sentiment_label} ({confidence_score:.1f}%)"
            else:
                row_record[aspect] = "░ Abstain (0.0%)"

        records.append(row_record)

    ordered_columns = ["Parsed Sentence Context"] + ASPECTS
    return pd.DataFrame(records, columns=ordered_columns)

# =====================================================================
# 🎨 STEP 3: THE MINIMALIST MONOCHROME INTERFACE LAYOUT
# =====================================================================

with gr.Blocks(theme=gr.themes.Monochrome(font=[gr.themes.GoogleFont("Inter"), "sans-serif"])) as demo:
    gr.Markdown("# ECONOMIC NEWS MULTI-ASPECT SENTIMENT SCORER")
    gr.Markdown("---")

    with gr.Tabs():
        with gr.TabItem("🎯 Aspect Classification"):
            gr.Markdown("### Evaluates whether sentences breach custom optimized DeBERTa thresholds.")
            input_text_p1 = gr.Textbox(label="Economic Input Headline / Clause", placeholder="Type a sentence here...")
            route_btn = gr.Button("Analyze System Routing Channels", variant="primary")
            output_df_p1 = gr.DataFrame(label="Calibrated Threshold Routing Status Matrix")

            route_btn.click(fn=process_routing, inputs=input_text_p1, outputs=output_df_p1)

        with gr.TabItem("⚖️ Aspect Sentiment Scoring"):
            gr.Markdown("### Compares Native Global FinBERT against your Fine-Tuned text-pair ABSA weights.")
            with gr.Row():
                input_text_p2 = gr.Textbox(label="Target Sentence Context", placeholder="Type targeted phrase here...")
                input_aspect_p2 = gr.Dropdown(choices=ASPECTS, label="Isolate Specific Evaluation Target Aspect", value=ASPECTS[1])

            score_btn = gr.Button("Execute Dual Inferences", variant="primary")

            with gr.Row():
                output_label_base = gr.Label(label="Baseline FinBERT (Net Global Sentence View)")
                output_label_tuned = gr.Label(label="Fine-Tuned ABSA Model (Targeted Context Pair View)")

            score_btn.click(fn=process_sentiment, inputs=[input_text_p2, input_aspect_p2], outputs=[output_label_base, output_label_tuned])

        with gr.TabItem("🚀 Production Document Parsing Engine"):
            gr.Markdown("### Paste a comprehensive macroeconomic article block or raw HTML source code below.")
            input_doc = gr.Textbox(label="Raw Market Article / Page Source Stream", lines=10, placeholder="Paste data here...")
            parse_btn = gr.Button("Parse Document Structure", variant="primary")
            output_df_p3 = gr.DataFrame(label="Decoded Multi-Label Executive Log Table", wrap=True)

            parse_btn.click(fn=process_document, inputs=input_doc, outputs=output_df_p3)

# =====================================================================
# 🏁 STEP 4: FLIP INLINE OFF & SHIFT TO A PUBLIC SECURE TUNNEL LINK
# =====================================================================
demo.launch(inline=False, share=True, debug=True)

📥 Initializing deep learning architectures straight from Hugging Face Hub...
🖥️ Routing computing matrix to: CUDA


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

🚀 Core neural engines loaded successfully!


/tmp/ipykernel_1288/1905175832.py:229: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Monochrome(font=[gr.themes.GoogleFont("Inter"), "sans-serif"])) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://50fc4bac8cda019e1c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://50fc4bac8cda019e1c.gradio.live
